In [1]:
import sys
sys.path.append('../')

In [2]:
import pandas as pd

In [3]:
from transform import get_state_map_long_to_short, transform_gen
from load_dims import get_engine, get_all_maps

In [24]:
from transform import transform_prices

In [ ]:
"""
WARNING: gen: missing fuel_id
  WARNING: prices: missing sector_id
"""

In [4]:
engine = get_engine()

Connection successful: (1,)


In [5]:
maps = get_all_maps(engine)

In [6]:
maps["fuel"]

{'ALL': 1,
 'AOR': 2,
 'BIO': 3,
 'COL': 4,
 'COW': 5,
 'DFO': 6,
 'FOS': 7,
 'GEO': 8,
 'HPS': 9,
 'HYC': 10,
 'LFG': 11,
 'LIG': 12,
 'MLG': 13,
 'NG': 14,
 'NGO': 15,
 'NUC': 16,
 'OOG': 17,
 'ORW': 18,
 'OTH': 19,
 'PEL': 20,
 'PET': 21,
 'REN': 22,
 'RFO': 23,
 'SPV': 24,
 'SUB': 25,
 'SUN': 26,
 'WAS': 27,
 'WND': 28,
 'WNT': 29,
 'WOC': 30,
 'WOO': 31,
 'WWW': 32,
 'MSB': 33,
 'PC': 34,
 'RC': 35,
 'STH': 36,
 'BIS': 37,
 'BIT': 38,
 'DPV': 39,
 'TPV': 40,
 'TSN': 41,
 'ANT': 42,
 'WNS': 43}

In [8]:
price_df = pd.read_csv("../data/price_data.csv")
price_df.head()

,period,stateid,stateDescription,sectorid,sectorName,price,price-units
0,2025-03,AK,Alaska,ALL,all sectors,22.96,cents per kilowatt-hour
1,2025-03,AK,Alaska,COM,commercial,22.10,cents per kilowatt-hour
2,2025-03,AK,Alaska,IND,industrial,20.17,cents per kilowatt-hour
3,2025-03,AK,Alaska,OTH,other,NaN,cents per kilowatt-hour
4,2025-03,AK,Alaska,RES,residential,25.79,cents per kilowatt-hour


In [7]:
gen_df = pd.read_csv("../data/gen_data.csv")
gen_df.head()

,period,location,stateDescription,sectorid,sectorDescription,fueltypeid,fuelTypeDescription,generation,generation-units
0,2025-03,90,Pacific,1,Electric Utility,ALL,all fuels,17842.73950,thousand megawatthours
1,2025-03,90,Pacific,1,Electric Utility,AOR,all renewables,822.00528,thousand megawatthours
2,2025-03,90,Pacific,1,Electric Utility,BIO,biomass,37.41079,thousand megawatthours
3,2025-03,90,Pacific,1,Electric Utility,COL,"coal, excluding waste coal",36.42772,thousand megawatthours
4,2025-03,90,Pacific,1,Electric Utility,COW,all coal products,36.42772,thousand megawatthours


In [9]:
long_to_short = get_state_map_long_to_short(price_df)

In [10]:
gen_clean = transform_gen(gen_df, long_to_short, maps)

In [11]:
gen_clean.head()

,date_id,state_id,fuel_id,generation
416,202503,1,1.0,410.46621
417,202503,1,2.0,9.51077
418,202503,1,4.0,36.42772
419,202503,1,5.0,36.42772
420,202503,1,6.0,47.27726


In [13]:
gen_df["fuel_id"] = gen_df.fueltypeid.map(maps["fuel"])
gen_df.head()

,period,location,stateDescription,sectorid,sectorDescription,fueltypeid,fuelTypeDescription,generation,generation-units,fuel_id
0,2025-03,90,Pacific,1,Electric Utility,ALL,all fuels,17842.73950,thousand megawatthours,1.0
1,2025-03,90,Pacific,1,Electric Utility,AOR,all renewables,822.00528,thousand megawatthours,2.0
2,2025-03,90,Pacific,1,Electric Utility,BIO,biomass,37.41079,thousand megawatthours,3.0
3,2025-03,90,Pacific,1,Electric Utility,COL,"coal, excluding waste coal",36.42772,thousand megawatthours,4.0
4,2025-03,90,Pacific,1,Electric Utility,COW,all coal products,36.42772,thousand megawatthours,5.0


In [14]:
gen_nulls = gen_df.loc[gen_df["fuel_id"].isnull()]
len(gen_nulls)

2298

In [18]:
gen_nulls.fuelTypeDescription.unique()

<StringArray>
['biomass']
Length: 1, dtype: str

In [19]:
gen_nulls.head()

,period,location,stateDescription,sectorid,sectorDescription,fueltypeid,fuelTypeDescription,generation,generation-units,fuel_id
16,2025-03,90,Pacific,1,Electric Utility,OB2,biomass,3.70237,thousand megawatthours,NaN
17,2025-03,90,Pacific,1,Electric Utility,OBW,biomass,3.70237,thousand megawatthours,NaN
48,2025-03,90,Pacific,2,IPP Non-CHP,OB2,biomass,30.73068,thousand megawatthours,NaN
49,2025-03,90,Pacific,2,IPP Non-CHP,OBW,biomass,36.37543,thousand megawatthours,NaN
80,2025-03,90,Pacific,3,IPP CHP,OB2,biomass,NaN,thousand megawatthours,NaN


In [20]:
price_df.head()

,period,stateid,stateDescription,sectorid,sectorName,price,price-units
0,2025-03,AK,Alaska,ALL,all sectors,22.96,cents per kilowatt-hour
1,2025-03,AK,Alaska,COM,commercial,22.10,cents per kilowatt-hour
2,2025-03,AK,Alaska,IND,industrial,20.17,cents per kilowatt-hour
3,2025-03,AK,Alaska,OTH,other,NaN,cents per kilowatt-hour
4,2025-03,AK,Alaska,RES,residential,25.79,cents per kilowatt-hour


In [22]:
price_df["sector_id"] = price_df.sectorName.map(maps["sector"])
price_df.head()

,period,stateid,stateDescription,sectorid,sectorName,price,price-units,sector_id
0,2025-03,AK,Alaska,ALL,all sectors,22.96,cents per kilowatt-hour,1
1,2025-03,AK,Alaska,COM,commercial,22.10,cents per kilowatt-hour,2
2,2025-03,AK,Alaska,IND,industrial,20.17,cents per kilowatt-hour,3
3,2025-03,AK,Alaska,OTH,other,NaN,cents per kilowatt-hour,4
4,2025-03,AK,Alaska,RES,residential,25.79,cents per kilowatt-hour,5


In [23]:
price_nulls = price_df.loc[price_df["sector_id"].isnull()]
len(price_nulls)

0

In [25]:
prices_clean = transform_prices(price_df, maps)
prices_clean.head()

,date_id,state_id,sector_id,price_per_kwh
0,202503,1,NaN,22.96
1,202503,1,NaN,22.10
2,202503,1,NaN,20.17
3,202503,1,NaN,NaN
4,202503,1,NaN,25.79


In [26]:
prices_clean.sector_id.unique()

array([nan])

In [27]:
maps["sector"]

{'all sectors': 1,
 'commercial': 2,
 'industrial': 3,
 'other': 4,
 'residential': 5,
 'transportation': 6}